In [ ]:
import numpy as np 
import pandas as pd 
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "/kaggle/input/cleaned-blog-authorship-corpus/dataset.csv"

df = pd.read_csv(file_path)

print("First 5 records:", df.head())

First 5 records:         id gender  age            industry  \
0  3581210   male   33  Finance & Property   
1  3581210   male   33  Finance & Property   
2  3581210   male   33  Finance & Property   
3  3581210   male   33  Finance & Property   
4  3581210   male   33  Finance & Property   

                                                text  word_count  \
0  i had an interesting conversation with my dad ...         662   
1  if anything, korea is a country of extremes. e...         387   
2  take a read of this news article from <url> jo...         386   
3  ah, the korean language...<ELONG>it looks so d...         296   
4  if you click on my profile you'll make a not-s...         499   

        age_group  
0  Group3 (30–48)  
1  Group3 (30–48)  
2  Group3 (30–48)  
3  Group3 (30–48)  
4  Group3 (30–48)  


In [ ]:
df.columns

Index(['id', 'gender', 'age', 'industry', 'text', 'word_count', 'age_group'], dtype='object')

In [ ]:
df.describe()

,id,age,word_count
count,9.619900e+04,96199.000000,96199.000000
mean,2.498590e+06,26.257113,450.997245
std,1.172653e+06,7.559463,245.166679
min,7.596000e+03,13.000000,178.000000
25%,1.476382e+06,23.000000,269.000000
50%,2.788090e+06,25.000000,373.000000
75%,3.513619e+06,27.000000,552.000000
max,4.332352e+06,48.000000,1500.000000


In [ ]:
list(set(df['industry']))

['Industrial & Misc.',
 'Arts',
 'Creative Media & Culture',
 'Public Service & Governance',
 'Education',
 'Science & Technical',
 'Business & Consulting',
 'Law & Specialized Services',
 'Non-Profit',
 'Technology',
 'Communications-Media',
 'Student',
 'Finance & Property',
 'Internet']

In [ ]:
!pip -q install transformers==4.43.3 accelerate==0.33.0

import os, random, math, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Config
SEED = 42
BATCH_TRAIN = 32
BATCH_VAL = 32
EPOCHS = 20
PATIENCE = 3
LR = 2e-5
MAX_LEN = 500
MODEL_NAME = "microsoft/deberta-v3-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 56.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 18.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:562: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
from transformers import DataCollatorWithPadding
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_age = LabelEncoder()
le_industry = LabelEncoder()

df["gender_lbl"]   = le_gender.fit_transform(df["gender"])
df["age_group_lbl"] = le_age.fit_transform(df["age_group"])
df["industry_lbl"]  = le_industry.fit_transform(df["industry"])

class MTDataset(Dataset):
    def __init__(self, df, max_len=256):
        self.texts = df["text"].astype(str).tolist()
        self.g = df["gender_lbl"].astype(int).tolist()
        self.a = df["age_group_lbl"].astype(int).tolist()
        self.i = df["industry_lbl"].astype(int).tolist()
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,   
            return_tensors="pt"        
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["g"] = torch.tensor(self.g[idx], dtype=torch.float32)
        item["a"] = torch.tensor(self.a[idx], dtype=torch.long)
        item["i"] = torch.tensor(self.i[idx], dtype=torch.long)
        return item

collate_fn = DataCollatorWithPadding(tokenizer=tok, return_tensors="pt")

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df["industry_lbl"]
)

ds_tr, ds_va = MTDataset(train_df, max_len=500), MTDataset(val_df, max_len=500)

dl_tr = DataLoader(
    ds_tr, batch_size=BATCH_TRAIN, shuffle=True,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)
dl_va = DataLoader(
    ds_va, batch_size=BATCH_VAL, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True, persistent_workers=True
)

num_age = df["age_group_lbl"].nunique()
num_ind = df["industry_lbl"].nunique()

In [ ]:
# Model
class MTLModel(nn.Module):
    def __init__(self, base, n_age, n_ind):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base)
        hid = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head_g = nn.Linear(hid, 1)
        self.head_a = nn.Linear(hid, n_age)
        self.head_i = nn.Linear(hid, n_ind)
        self.bce = nn.BCEWithLogitsLoss()
        self.ce_a = nn.CrossEntropyLoss(label_smoothing=0.05)
        self.ce_i = nn.CrossEntropyLoss(label_smoothing=0.05)

    def forward(self, input_ids, attention_mask, g=None, a=None, i=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = self.dropout(out.last_hidden_state[:,0])  
        lg = self.head_g(h).squeeze(1)
        la = self.head_a(h)
        li = self.head_i(h)
        losses = None
        if g is not None:
            Lg = self.bce(lg, g)
            La = self.ce_a(la, a)
            Li = self.ce_i(li, i)
            # loss = Lg + La + 1.5*Li  
            loss = Lg + La + Li  
            losses = (loss, Lg.detach(), La.detach(), Li.detach())
        return (lg, la, li), losses

# Training setup
model = MTLModel(MODEL_NAME, num_age, num_ind).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_train_steps = EPOCHS * len(dl_tr)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=num_train_steps)

# Evaluation
def evaluate():
    model.eval()
    g_probs, g_true, a_pred, a_true, i_pred, i_true = [], [], [], [], [], []
    with torch.no_grad():
        for batch in dl_va:
            for k in ("input_ids","attention_mask"): batch[k] = batch[k].to(device)
            g, a, i = batch["g"].to(device), batch["a"].to(device), batch["i"].to(device)
            (lg, la, li), _ = model(batch["input_ids"], batch["attention_mask"])
            g_probs.extend(torch.sigmoid(lg).cpu().numpy())
            g_true.extend(g.cpu().numpy())
            a_pred.extend(la.argmax(1).cpu().numpy()); a_true.extend(a.cpu().numpy())
            i_pred.extend(li.argmax(1).cpu().numpy()); i_true.extend(i.cpu().numpy())
    g_probs = np.array(g_probs); g_true = np.array(g_true)
    g_pred = (g_probs >= 0.5).astype(int)
    return {
        "G_acc": accuracy_score(g_true, g_pred),
        "G_f1": f1_score(g_true, g_pred, average="macro"),
        "A_acc": accuracy_score(a_true, a_pred),
        "A_f1": f1_score(a_true, a_pred, average="macro"),
        "I_acc": accuracy_score(i_true, i_pred),
        "I_f1": f1_score(i_true, i_pred, average="macro"),
    }

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [ ]:
best_score = -1
best_state = None
no_improve = 0

for ep in range(1, EPOCHS+1):
    model.train()
    tr_losses = []
    pbar = tqdm(dl_tr, desc=f"Epoch {ep}", leave=False)
    for batch in pbar:
        for k in ("input_ids","attention_mask"): batch[k] = batch[k].to(device)
        g, a, i = batch["g"].to(device), batch["a"].to(device), batch["i"].to(device)
        (_,_,_), losses = model(batch["input_ids"], batch["attention_mask"], g,a,i)
        loss = losses[0]
        if loss.dim() > 0:       
            loss = loss.mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        tr_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    metrics = evaluate()
    comp = (metrics["G_f1"] + metrics["A_f1"] + metrics["I_f1"]) / 3.0
    print(f"Epoch {ep:02d} | train {np.mean(tr_losses):.4f} | "
          f"G {metrics['G_acc']:.3f}/{metrics['G_f1']:.3f} | "
          f"A {metrics['A_acc']:.3f}/{metrics['A_f1']:.3f} | "
          f"I {metrics['I_acc']:.3f}/{metrics['I_f1']:.3f} | comp {comp:.3f}")

    # Early stopping
    if comp > best_score:
        best_score = comp
        best_state = {k:v.cpu() for k,v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping.")
            break

# Load best checkpoint
if best_state:
    model.load_state_dict(best_state)
    model.to(device)

Epoch 1:   0%|          | 0/2556 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch 01 | train 4.9049 | G 0.763/0.761 | A 0.711/0.695 | I 0.280/0.213 | comp 0.556


Epoch 2:   0%|          | 0/2556 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


...

Epoch 3:   0%|          | 0/2556 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch 03 | train 3.9331 | G 0.770/0.770 | A 0.762/0.730 | I 0.361/0.327 | comp 0.609


Epoch 4:   0%|          | 0/2556 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [ ]:
# Save the best model
torch.save(model.state_dict(), "/kaggle/working/mtl_model_best.pth")

import joblib
joblib.dump(le_gender, "/kaggle/working/le_gender.pkl")
joblib.dump(le_age, "/kaggle/working/le_age.pkl")
joblib.dump(le_industry, "/kaggle/working/le_industry.pkl")
